## Exploratory Data Analysis of Dataset 1 - Fake News
---

### Import Relevant Libraries

In [30]:
# For dataset manipulation
import numpy as np
import pandas as pd
# For punctuation count
import spacy
# For readability ease count
import textstat
# For progress bar
from tqdm.notebook import tqdm
tqdm.pandas()

### Load The spaCy English Model

In [31]:
nlp = spacy.load("en_core_web_sm")

### Custom Functions

In [ ]:
# Removes whitespaces and lowers case
def whitespace_lower(component):
    return component.str.lower().str.split().str.join(' ')

# Calculate total words
def calc_word(component):
    return component.astype(str).apply(lambda x: len(x.split()))

# Calculate total punctuations
def calc_punct(component):
    doc = nlp(component)
    # For every punct found, sum increases by 1
    return sum(1 for token in doc if token.is_punct) 

# Calculate readability ease
def calc_readability(component):
    return textstat.flesch_reading_ease(component)


### Fundamental Data Analysis

In [33]:
# Read the fake news dataset from Dataset 1 and store it in df_fake
df_fake = pd.read_csv("../../Datasets/Dataset1_fake.csv")

In [34]:
# General overview
df_fake_info = df_fake.info()
df_fake_info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [35]:
# Amount of unique values per column
df_fake_unique_values = df_fake.nunique()
df_fake_unique_values

title      17903
text       17455
subject        6
date        1681
dtype: int64

In [36]:
# Total number of NaN (null) values
df_fake_total_null_values = df_fake.isna().sum()
df_fake_total_null_values

title      0
text       0
subject    0
date       0
dtype: int64

In [37]:
# Total number of duplicate values for title
df_fake_title_duplicate_count = df_fake['title'].duplicated(keep="first").sum() # keep="first" ensures the first occurence of the duplicate is kept
df_fake_title_duplicate_count

5578

In [38]:
# Total number of duplicate values for text
df_fake_text_duplicate_count = df_fake['text'].duplicated(keep='first').sum()
df_fake_text_duplicate_count

6026

In [39]:
# Total number of combined duplicate values
df_fake_combined_duplicate_count = df_fake.duplicated(subset=['title', 'text']).sum()
df_fake_combined_duplicate_count

5573

### Drop Combined Raw Duplicates 

In [40]:
df_fake_no_raw_duplicates = df_fake.drop_duplicates(subset=['title', 'text'], keep='first').reset_index(drop=True)

In [41]:
df_fake_no_raw_duplicates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17908 entries, 0 to 17907
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    17908 non-null  object
 1   text     17908 non-null  object
 2   subject  17908 non-null  object
 3   date     17908 non-null  object
dtypes: object(4)
memory usage: 559.8+ KB


### Dataset Rearrangement 1 (Title, Text, Label)

In [42]:
# Drop the non-required columns, "subject" and "date"
df_fake_no_raw_duplicates = df_fake_no_raw_duplicates.drop(columns=["subject", "date"]).reset_index(drop=True)

# Add a column, "label" where everything is 0 = Fake
df_fake_no_raw_duplicates["label"] = 0

# Show the dataset
df_fake_no_raw_duplicates

,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,0
...,...,...,...
17903,The White House and The Theatrics of ‘Gun Cont...,21st Century Wire says All the world s a stage...,0
17904,Activists or Terrorists? How Media Controls an...,Randy Johnson 21st Century WireThe majority ...,0
17905,"BOILER ROOM – No Surrender, No Retreat, Heads ...",Tune in to the Alternate Current Radio Network...,0
17906,Federal Showdown Looms in Oregon After BLM Abu...,21st Century Wire says A new front has just op...,0


### Word, Punctuation and Readability Ease Counts

In [ ]:
# Count the number of words, punctuations and the ease in redability for title
df_fake_no_raw_duplicates["title_word_count"] = calc_word(df_fake_no_raw_duplicates["title"])
df_fake_no_raw_duplicates["title_punct_count"] = df_fake_no_raw_duplicates["title"].progress_apply(calc_punct) # .progress to trigger progress bar
df_fake_no_raw_duplicates["title_readability_ease"] = df_fake_no_raw_duplicates["title"].progress_apply(calc_readability)
 

  0%|          | 0/17908 [00:00<?, ?it/s]

  0%|          | 0/17908 [00:00<?, ?it/s]

In [ ]:
# Count the number of words, punctuations and the ease in redability for text
df_fake_no_raw_duplicates["text_word_count"] = calc_word(df_fake_no_raw_duplicates["text"])
df_fake_no_raw_duplicates["text_punct_count"] = df_fake_no_raw_duplicates["text"].progress_apply(calc_punct)
df_fake_no_raw_duplicates["text_readability_ease"] = df_fake_no_raw_duplicates["text"].progress_apply(calc_readability)

  0%|          | 0/17908 [00:00<?, ?it/s]

  0%|          | 0/17908 [00:00<?, ?it/s]

### Dataset Rearrangement 2 (Added On Word, Punctuation and Readability Ease)


In [45]:
# Rearranging column order from left to right
df_fake_no_raw_duplicates = df_fake_no_raw_duplicates[["title", "title_word_count", "title_punct_count", "title_readability_ease", "text", "text_word_count", "text_punct_count", "text_readability_ease", "label"]]

# Show the updated dataset
df_fake_no_raw_duplicates

,title,title_word_count,title_punct_count,title_readability_ease,text,text_word_count,text_punct_count,text_readability_ease,label
0,Donald Trump Sends Out Embarrassing New Year’...,12,1,59.30,Donald Trump just couldn t wish all Americans ...,495,101,62.68,0
1,Drunk Bragging Trump Staffer Started Russian ...,8,0,4.14,House Intelligence Committee Chairman Devin Nu...,305,34,45.86,0
2,Sheriff David Clarke Becomes An Internet Joke...,15,2,64.71,"On Friday, it was revealed that former Milwauk...",580,103,63.59,0
3,Trump Is So Obsessed He Even Has Obama’s Name...,14,2,57.27,"On Christmas day, Donald Trump announced that ...",444,66,53.61,0
4,Pope Francis Just Called Out Donald Trump Dur...,11,0,77.23,Pope Francis used his annual Christmas Day mes...,420,40,58.62,0
...,...,...,...,...,...,...,...,...,...
17903,The White House and The Theatrics of ‘Gun Cont...,9,2,96.18,21st Century Wire says All the world s a stage...,1226,166,37.74,0
17904,Activists or Terrorists? How Media Controls an...,13,4,39.50,Randy Johnson 21st Century WireThe majority ...,4257,547,37.54,0
17905,"BOILER ROOM – No Surrender, No Retreat, Heads ...",13,5,77.23,Tune in to the Alternate Current Radio Network...,183,29,36.73,0
17906,Federal Showdown Looms in Oregon After BLM Abu...,16,1,30.87,21st Century Wire says A new front has just op...,3480,421,50.87,0


### Save The Updated Dataset

In [47]:
df_fake_no_raw_duplicates.to_csv('../../Datasets/Dataset1_fake_counts_cleaned_1.csv', index=False)

### Data Rearrangement 3 (Combine Title + Text)

In [65]:
df_fake_combined = pd.read_csv("../../Datasets/Dataset1_fake_counts_cleaned_1.csv")

In [67]:
df_fake_combined['combined_data'] = df_fake_combined['title'] + ' ' + df_fake_combined['text']

In [69]:
df_fake_combined = df_fake_combined.drop(columns=[
    'title',
    'title_word_count', 
    'title_punct_count',
    'title_readability_ease',
    'text',
    'text_word_count', 
    'text_punct_count',
    'text_readability_ease'
]).reset_index(drop=True)

### Dataset Rearrangement 4 (Added On Combined Word, Punctuation and Readability Ease)

In [73]:
# Count the number of words, punctuations and the ease in redability for combined data
df_fake_combined["combined_word_count"] = calc_word(df_fake_combined["combined_data"])
df_fake_combined["combined_punct_count"] = df_fake_combined["combined_data"].progress_apply(calc_punct)
df_fake_combined["combined_readability_ease"] = df_fake_combined["combined_data"].progress_apply(calc_readability)

  0%|          | 0/17908 [00:00<?, ?it/s]

  0%|          | 0/17908 [00:00<?, ?it/s]

In [75]:
df_fake_combined = df_fake_combined[["combined_data", "combined_word_count", "combined_punct_count", "combined_readability_ease", "label"]]
df_fake_combined

,combined_data,combined_word_count,combined_punct_count,combined_readability_ease,label
0,Donald Trump Sends Out Embarrassing New Year’...,507,102,62.27,0
1,Drunk Bragging Trump Staffer Started Russian ...,313,34,45.35,0
2,Sheriff David Clarke Becomes An Internet Joke...,595,105,63.19,0
3,Trump Is So Obsessed He Even Has Obama’s Name...,458,68,53.00,0
4,Pope Francis Just Called Out Donald Trump Dur...,431,40,58.01,0
...,...,...,...,...,...
17903,The White House and The Theatrics of ‘Gun Cont...,1235,168,46.10,0
17904,Activists or Terrorists? How Media Controls an...,4270,551,37.64,0
17905,"BOILER ROOM – No Surrender, No Retreat, Heads ...",196,34,35.20,0
17906,Federal Showdown Looms in Oregon After BLM Abu...,3496,422,50.77,0


### Save the Updated Dataset

In [76]:
df_fake_combined.to_csv('../../Datasets/Dataset1_fake_combined_counts_cleaned_1.csv', index=False)